<a href="https://colab.research.google.com/github/NMH1203/DL2026-13-18/blob/main/Notebooks/NDN/Exdark_Yolo_Clahe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%pip install -q opencv-python-headless tqdm pyyaml

In [ ]:
!git clone https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git /content/Dataset

Cloning into '/content/Dataset'...
remote: Enumerating objects: 15130, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 15130 (delta 5), reused 58 (delta 5), pack-reused 15064 (from 2)
Receiving objects: 100% (15130/15130), 439.75 MiB | 12.55 MiB/s, done.
Resolving deltas: 100% (72/72), done.
Updating files: 100% (14939/14939), done.


In [ ]:
import os

print("Repo exists:", os.path.isdir("/content/Dataset"))
print("train_ninh.py exists:",
      os.path.isfile("/content/Dataset/src/LBN/train_ninh.py"))

Repo exists: True
train_ninh.py exists: True


In [ ]:
from pathlib import Path

# Original ExDark YOLO dataset from the GitHub repository
SOURCE_DATASET = (
    Path("/content/Dataset")
    / "Dataset"
    / "exdark_yolo_dark"
)

# New CLAHE-only dataset saved to Google Drive
OUTPUT_DATASET = Path(
    "/content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only"
)

SPLITS = ["train", "valid", "test"]

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}

print("Source dataset:", SOURCE_DATASET)
print("Output dataset:", OUTPUT_DATASET)

print("\nSource dataset exists:", SOURCE_DATASET.exists())

if SOURCE_DATASET.exists():
    for split in SPLITS:
        image_dir = SOURCE_DATASET / split / "images"
        label_dir = SOURCE_DATASET / split / "labels"

        print(
            f"{split}: "
            f"images={image_dir.exists()}, "
            f"labels={label_dir.exists()}"
        )

assert SOURCE_DATASET.exists(), (
    f"Dataset not found: {SOURCE_DATASET}"
)

Source dataset: /content/Dataset/Dataset/exdark_yolo_dark
Output dataset: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only

Source dataset exists: True
train: images=True, labels=True
valid: images=True, labels=True
test: images=True, labels=True


In [ ]:
"""CLAHE-only enhancement for the ExDark dataset."""

import cv2
import numpy as np


def enhance_clahe(image_bgr: np.ndarray) -> np.ndarray:
    """Increase local luminance contrast using CLAHE."""
    if image_bgr is None or image_bgr.size == 0:
        raise ValueError("Input image is empty.")

    lab = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2LAB,
    )

    luminance, channel_a, channel_b = cv2.split(lab)

    luminance = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8),
    ).apply(luminance)

    enhanced = cv2.cvtColor(
        cv2.merge(
            (
                luminance,
                channel_a,
                channel_b,
            )
        ),
        cv2.COLOR_LAB2BGR,
    )

    return enhanced

In [ ]:
import cv2
import shutil

from tqdm.auto import tqdm

total_processed = 0
total_skipped = 0

for split in SPLITS:
    source_images_dir = SOURCE_DATASET / split / "images"
    source_labels_dir = SOURCE_DATASET / split / "labels"

    output_images_dir = OUTPUT_DATASET / split / "images"
    output_labels_dir = OUTPUT_DATASET / split / "labels"

    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_labels_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(
        path
        for path in source_images_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    label_paths = sorted(source_labels_dir.glob("*.txt"))

    print(
        f"{split}: {len(image_paths)} images, "
        f"{len(label_paths)} labels"
    )

    # Copy original YOLO labels
    # CLAHE does not resize or crop the image,
    # so bounding boxes remain unchanged.
    for label_path in label_paths:
        shutil.copy2(
            label_path,
            output_labels_dir / label_path.name,
        )

    # Process images sequentially
    for image_path in tqdm(
        image_paths,
        desc=f"CLAHE-only {split}",
    ):
        output_path = (
            output_images_dir
            / f"{image_path.stem}.png"
        )

        # Allow the process to continue after an interruption.
        if output_path.exists() and output_path.stat().st_size > 0:
            total_skipped += 1
            continue

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            raise ValueError(
                f"Cannot read image: {image_path}"
            )

        original_shape = image_bgr.shape

        enhanced_bgr = enhance_clahe(image_bgr)

        if enhanced_bgr.shape != original_shape:
            raise ValueError(
                f"Image size changed: {image_path}"
            )

        success = cv2.imwrite(
            str(output_path),
            enhanced_bgr,
            [cv2.IMWRITE_PNG_COMPRESSION, 3],
        )

        if not success:
            raise IOError(
                f"Cannot save image: {output_path}"
            )

        total_processed += 1

print("\nCLAHE-only processing completed")
print("New images processed:", total_processed)
print("Existing images skipped:", total_skipped)

train: 5142 images, 5142 labels


CLAHE-only train:   0%|          | 0/5142 [00:00<?, ?it/s]

valid: 1469 images, 1469 labels


CLAHE-only valid:   0%|          | 0/1469 [00:00<?, ?it/s]

test: 734 images, 734 labels


CLAHE-only test:   0%|          | 0/734 [00:00<?, ?it/s]


CLAHE-only processing completed
New images processed: 7345
Existing images skipped: 0


In [ ]:
import yaml
from pathlib import Path

DATA_YAML = OUTPUT_DATASET / "data.yaml"

CLASS_NAMES = [
    "Bicycle",
    "Boat",
    "Bottle",
    "Bus",
    "Cat",
    "Cup",
    "Motorbike",
    "People",
    "Table",
    "car",
    "chair",
    "dog",
]

data_config = {
    "path": str(OUTPUT_DATASET),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": {
        index: name
        for index, name in enumerate(CLASS_NAMES)
    },
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False,
        allow_unicode=True,
    )

print("========== DATA.YAML ==========")
print(DATA_YAML)
print()
print(yaml.dump(data_config, sort_keys=False))
print("data.yaml exists:", DATA_YAML.exists())

========== DATA.YAML ==========
/content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only/data.yaml

path: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only
train: train/images
val: valid/images
test: test/images
nc: 12
names:
  0: Bicycle
  1: Boat
  2: Bottle
  3: Bus
  4: Cat
  5: Cup
  6: Motorbike
  7: People
  8: Table
  9: car
  10: chair
  11: dog

data.yaml exists: True


In [ ]:
import os
import yaml

print("========== DATASET CHECK ==========")
print("Dataset:", OUTPUT_DATASET)
print("data.yaml:", DATA_YAML)
print()

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("========== YAML ==========")
print(yaml.dump(config, sort_keys=False))

all_ok = True

for split in SPLITS:
    image_dir = OUTPUT_DATASET / split / "images"
    label_dir = OUTPUT_DATASET / split / "labels"

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = [
        p for p in label_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    ]

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems

    print(f"\n========== {split.upper()} ==========")
    print("Images :", len(images))
    print("Labels :", len(labels))
    print("Missing labels:", len(missing_labels))
    print("Missing images:", len(missing_images))

    if missing_labels or missing_images:
        all_ok = False

print("\n========== LABEL FORMAT CHECK ==========")

bad_format = 0
bad_class = 0
bad_coordinate = 0
total_objects = 0

for split in SPLITS:
    label_dir = OUTPUT_DATASET / split / "labels"

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 5:
                bad_format += 1
                continue

            try:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(
                    float,
                    parts[1:]
                )
            except ValueError:
                bad_format += 1
                continue

            total_objects += 1

            if not 0 <= class_id < len(CLASS_NAMES):
                bad_class += 1

            if not all(
                0 <= value <= 1
                for value in (
                    x_center,
                    y_center,
                    width,
                    height,
                )
            ):
                bad_coordinate += 1

print("Total objects checked :", total_objects)
print("Bad label format      :", bad_format)
print("Bad class ID           :", bad_class)
print("Bad coordinates        :", bad_coordinate)

print("\n========== FINAL RESULT ==========")

if all_ok and bad_format == 0 and bad_class == 0 and bad_coordinate == 0:
    print("✅ DATASET IS READY FOR YOLO TRAINING")
else:
    print("❌ DATASET HAS PROBLEMS - DO NOT TRAIN YET")

========== DATASET CHECK ==========
Dataset: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only
data.yaml: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only/data.yaml

========== YAML ==========
path: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only
train: train/images
val: valid/images
test: test/images
nc: 12
names:
  0: Bicycle
  1: Boat
  2: Bottle
  3: Bus
  4: Cat
  5: Cup
  6: Motorbike
  7: People
  8: Table
  9: car
  10: chair
  11: dog


========== TRAIN ==========
Images : 5142
Labels : 5142
Missing labels: 0
Missing images: 0

========== VALID ==========
Images : 1469
Labels : 1469
Missing labels: 0
Missing images: 0

========== TEST ==========
Images : 734
Labels : 734
Missing labels: 0
Missing images: 0

========== LABEL FORMAT CHECK ==========
Total objects checked : 0
Bad label format      : 23146
Bad class ID           : 0
Bad coordinates        : 0

========== FINAL RESULT ==========
❌ DATASET HAS PROBLEMS - DO NOT TRAIN Y

In [ ]:
from pathlib import Path

print("========== SAMPLE LABEL FORMAT ==========")

shown = 0

for split in SPLITS:
    label_dir = SOURCE_DATASET / split / "labels"

    for label_path in sorted(label_dir.glob("*.txt")):
        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        if not lines:
            continue

        print(f"\nSplit : {split}")
        print(f"File  : {label_path.name}")
        print(f"Line  : {lines[0]}")
        print(f"Fields: {len(lines[0].split())}")

        shown += 1

        if shown >= 5:
            break

    if shown >= 5:
        break

========== SAMPLE LABEL FORMAT ==========

Split : train
File  : 2015_00001_png.rf.32acfca725ca0564c4807de73544a714.txt
Line  : 0 0.965625 0.10312500000000009 0.4734375 0.10312500000000005 0.4734375 0.5531250000000001 0.965625 0.5531250000000002
Fields: 9

Split : train
File  : 2015_00002_png.rf.da199d65b28766e41fb8dc7ed6db29e6.txt
Line  : 0 0.7156250000000001 0.5828125 0.575 0.5828125 0.5750000000000001 0.9484375 0.7156250000000002 0.9484375
Fields: 9

Split : train
File  : 2015_00003_png.rf.9b62b55d3d3bb51ab9c8739d6f7ebb1e.txt
Line  : 0 0.978125 0.30937500000000007 0.5859375 0.30937500000000007 0.5859375000000001 0.8359375 0.978125 0.8359375
Fields: 9

Split : train
File  : 2015_00004_jpg.rf.1afc46b07c6e89e6ea2b53c2523a95e6.txt
Line  : 0 0.7812500000000001 0.3546875000000001 0.5265625 0.3546875000000001 0.5265625 0.753125 0.7812500000000002 0.7531250000000002
Fields: 9

Split : train
File  : 2015_00005_jpg.rf.bbafa526d88f74c32f9a2bce2c87a522.txt
Line  : 0 0.6671875 0.1828125000000000

In [ ]:
from pathlib import Path

converted_labels = 0
copied_detection_labels = 0
empty_labels = 0
bad_labels = 0
total_objects = 0

for split in SPLITS:
    label_dir = OUTPUT_DATASET / split / "labels"

    for label_path in sorted(label_dir.glob("*.txt")):
        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        new_lines = []

        if not lines:
            empty_labels += 1
            continue

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            # Already standard YOLO detection format
            if len(parts) == 5:
                try:
                    class_id = int(parts[0])
                    values = list(map(float, parts[1:]))

                    if not 0 <= class_id < len(CLASS_NAMES):
                        raise ValueError(
                            f"Invalid class ID: {class_id}"
                        )

                    values = [
                        max(0.0, min(1.0, value))
                        for value in values
                    ]

                    new_lines.append(
                        f"{class_id} "
                        f"{values[0]:.6f} "
                        f"{values[1]:.6f} "
                        f"{values[2]:.6f} "
                        f"{values[3]:.6f}"
                    )

                    copied_detection_labels += 1
                    total_objects += 1

                except Exception as e:
                    print(
                        f"ERROR: {label_path.name}, "
                        f"line {line_number}: {e}"
                    )
                    bad_labels += 1

                continue

            # OBB format:
            # class x1 y1 x2 y2 x3 y3 x4 y4
            if len(parts) == 9:
                try:
                    class_id = int(parts[0])
                    points = list(map(float, parts[1:]))

                    if not 0 <= class_id < len(CLASS_NAMES):
                        raise ValueError(
                            f"Invalid class ID: {class_id}"
                        )

                    x_values = points[0::2]
                    y_values = points[1::2]

                    x_min = max(0.0, min(x_values))
                    y_min = max(0.0, min(y_values))
                    x_max = min(1.0, max(x_values))
                    y_max = min(1.0, max(y_values))

                    x_center = (x_min + x_max) / 2.0
                    y_center = (y_min + y_max) / 2.0
                    width = x_max - x_min
                    height = y_max - y_min

                    new_lines.append(
                        f"{class_id} "
                        f"{x_center:.6f} "
                        f"{y_center:.6f} "
                        f"{width:.6f} "
                        f"{height:.6f}"
                    )

                    converted_labels += 1
                    total_objects += 1

                except Exception as e:
                    print(
                        f"ERROR: {label_path.name}, "
                        f"line {line_number}: {e}"
                    )
                    bad_labels += 1

                continue

            print(
                f"BAD FORMAT: {label_path.name}, "
                f"line {line_number}, "
                f"fields={len(parts)}"
            )

            bad_labels += 1

        with open(label_path, "w") as f:
            if new_lines:
                f.write("\n".join(new_lines) + "\n")

print("========== CONVERSION RESULT ==========")
print("OBB objects converted       :", converted_labels)
print("Existing detection objects  :", copied_detection_labels)
print("Total objects                :", total_objects)
print("Empty label files            :", empty_labels)
print("Bad labels                   :", bad_labels)

if bad_labels == 0:
    print("\n✅ OBB → YOLO detection conversion completed.")
else:
    print("\n❌ Some labels could not be converted.")

========== CONVERSION RESULT ==========
OBB objects converted       : 23146
Existing detection objects  : 0
Total objects                : 23146
Empty label files            : 3
Bad labels                   : 0

✅ OBB → YOLO detection conversion completed.


In [ ]:
print("========== FINAL LABEL CHECK ==========")

bad_format = 0
bad_class = 0
bad_coordinate = 0
total_objects = 0
empty_label_files = 0

for split in SPLITS:
    label_dir = OUTPUT_DATASET / split / "labels"

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        if not lines:
            empty_label_files += 1
            continue

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 5:
                bad_format += 1
                continue

            try:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(
                    float,
                    parts[1:]
                )
            except ValueError:
                bad_format += 1
                continue

            total_objects += 1

            if not 0 <= class_id < len(CLASS_NAMES):
                bad_class += 1

            if not all(
                0.0 <= value <= 1.0
                for value in (
                    x_center,
                    y_center,
                    width,
                    height,
                )
            ):
                bad_coordinate += 1

print("Total objects      :", total_objects)
print("Empty label files  :", empty_label_files)
print("Bad format         :", bad_format)
print("Bad class ID       :", bad_class)
print("Bad coordinates    :", bad_coordinate)

print("\n========== RESULT ==========")

if (
    total_objects == 23146
    and empty_label_files == 3
    and bad_format == 0
    and bad_class == 0
    and bad_coordinate == 0
):
    print("✅ LABEL DATASET IS READY FOR YOLO")
else:
    print("❌ LABEL DATASET STILL HAS PROBLEMS")

========== FINAL LABEL CHECK ==========
Total objects      : 23146
Empty label files  : 3
Bad format         : 0
Bad class ID       : 0
Bad coordinates    : 0

========== RESULT ==========
✅ LABEL DATASET IS READY FOR YOLO


In [ ]:
%cd /content/Dataset

!pip install -q -U ultralytics

import os
import torch
import ultralytics

TRAIN_SCRIPT = "/content/Dataset/src/LBN/train_ninh.py"
MODEL_PATH = "/content/Dataset/yolov8n.pt"

print("========== ENVIRONMENT ==========")
print("Ultralytics version :", ultralytics.__version__)
print("PyTorch version     :", torch.__version__)
print("CUDA available      :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU                 :", torch.cuda.get_device_name(0))
    print("CUDA version        :", torch.version.cuda)

print("\n========== PROJECT FILES ==========")
print("train_ninh.py exists:", os.path.isfile(TRAIN_SCRIPT))
print("yolov8n.pt exists   :", os.path.isfile(MODEL_PATH))

if os.path.isfile(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)
    print(f"yolov8n.pt size      : {size_mb:.2f} MB")

/content/Dataset
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 455.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 10.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
========== ENVIRONMENT ==========
Ultralytics version : 8.4.173
PyTorch version     : 2.11.0+cu130
CUDA available      : True
GPU                 : Tesla T4
CUDA version        : 13.0

========== PROJECT FILES ==========
train_ninh.py exists: True
yolov8n.pt exists   : True
yolov8n.pt size      : 6.25 MB


In [ ]:
import os

RESULTS_ROOT = "/content/drive/MyDrive/Project18_Results"
RUN_NAME = "clahe_yolov8n"

RESULT_DIR = os.path.join(
    RESULTS_ROOT,
    RUN_NAME,
)

print("========== RESULT DIRECTORY ==========")
print("Results root exists:", os.path.isdir(RESULTS_ROOT))
print("Target run exists :", os.path.isdir(RESULT_DIR))
print("Target path       :", RESULT_DIR)

if os.path.isdir(RESULT_DIR):
    print("\nExisting files/folders:")
    for item in os.listdir(RESULT_DIR):
        print(" -", item)

========== RESULT DIRECTORY ==========
Results root exists: False
Target run exists : False
Target path       : /content/drive/MyDrive/Project18_Results/clahe_yolov8n


In [ ]:
%cd /content/Dataset

!python src/LBN/train_ninh.py train \
    --data /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only/data.yaml \
    --device 0 \
    --project /content/drive/MyDrive/Project18_Results \
    --name clahe_yolov8n \
    --epochs 40 \
    --batch 16 \
    --imgsz 640 \
    --seed 42 \
    --patience 15

/content/Dataset
Device=0, CUDA available=True
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, m

In [ ]:
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/Project18_Results/clahe_yolov8n"
)

print("========== TRAINING RESULT CHECK ==========")
print("Result directory exists:", RESULT_DIR.exists())
print()

for path in [
    RESULT_DIR / "weights" / "best.pt",
    RESULT_DIR / "weights" / "last.pt",
    RESULT_DIR / "results.csv",
    RESULT_DIR / "args.yaml",
]:
    print(f"{path.name:12}:", path.exists(), "|", path)

print()
print("Files in result directory:")
for item in sorted(RESULT_DIR.iterdir()):
    print("-", item.name)

========== TRAINING RESULT CHECK ==========
Result directory exists: True

best.pt     : True | /content/drive/MyDrive/Project18_Results/clahe_yolov8n/weights/best.pt
last.pt     : True | /content/drive/MyDrive/Project18_Results/clahe_yolov8n/weights/last.pt
results.csv : True | /content/drive/MyDrive/Project18_Results/clahe_yolov8n/results.csv
args.yaml   : True | /content/drive/MyDrive/Project18_Results/clahe_yolov8n/args.yaml

Files in result directory:
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- environment-freeze.txt
- environment.json
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- train_batch9660.jpg
- train_batch9661.jpg
- train_batch9662.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
GPU: None
